In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Library

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Flatten, Dense, Embedding, Dropout, Concatenate, MultiHeadAttention, Lambda
from tensorflow.keras.optimizers import Adam


from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [ ]:
def data_split(df, test_size=0.2, valid_size=0.125, random_state=42):

    # 1차 분할: train+valid, test
    df_train, df_test = train_test_split(df, test_size=test_size, random_state=random_state)

    # 2차 분할: train, valid
    df_train, df_valid = train_test_split(df_train, test_size=valid_size, random_state=random_state)

    df_train = df_train.reset_index(drop=True)
    df_valid = df_valid.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

    'Training Dataset'
    user_id_train = df_train['user_id'].values
    item_id_train = df_train['asin'].values
    roberta_train = np.array(df_train['roberta_emb'].tolist())
    vgg16_train = np.array(df_train['VGG16_emb'].tolist())
    y_train = df_train['rating'].values

    'Valid Dataset'
    user_id_valid = df_valid['user_id'].values
    item_id_valid = df_valid['asin'].values
    roberta_valid = np.array(df_valid['roberta_emb'].tolist())
    vgg16_valid = np.array(df_valid['VGG16_emb'].tolist())
    y_valid = df_valid['rating'].values

    'Test Dataset'
    user_id_test = df_test['user_id'].values
    item_id_test = df_test['asin'].values
    roberta_test = np.array(df_test['roberta_emb'].tolist())
    vgg16_test = np.array(df_test['VGG16_emb'].tolist())
    y_test = df_test['rating'].values

    return (
        user_id_train, item_id_train, roberta_train, vgg16_train, y_train,
        user_id_valid, item_id_valid, roberta_valid, vgg16_valid, y_valid,
        user_id_test, item_id_test, roberta_test, vgg16_test, y_test
    )

# Load Data

In [ ]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_baseline_emb.pkl')

print(df.shape)
df.head()

(66718, 15)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb,user_bert_emb,item_bert_emb,user_roberta_emb,item_roberta_emb
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 12.1029911041, 0.0, 15.7332105637, 4.268...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.20688048, -0.32638466, 0.13180715, -0.5702...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.06857138, 0.13921358, -0.039587017, -0.088..."
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 0.0, 2.2863385677, 10.4048433304, 6.9813...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.062450625, -0.07075825, 0.15572476, -0.2797...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.04617965, 0.12618569, -0.017711096, -0.102..."
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 37.5705032349, 2.7596354485, 0.0, 12.911...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.26902932, -0.42407188, 0.82339007, -0.2940...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.062135138, 0.1333665, -0.019405615, -0.102..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 18.6175479889, 0.0, 0.0, 3.5440309048, 0...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.119323395, -0.57359934, 0.40760195, -0.2408...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.06780286, 0.051634714, -0.017799914, -0.11..."
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[1.9380488396, 0.0, 0.0, 6.4230856895, 0.0, 0....","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.43734312, -0.62041384, 0.14694075, -0.3723...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.07941287, 0.05176169, -0.02554227, -0.0903..."


In [ ]:
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

# 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
print(n_users)
n_items = df['asin'].nunique()
print(n_items)

7449
16099


In [ ]:
user_id_train, item_id_train, roberta_train, vgg16_train, y_train, \
user_id_valid, item_id_valid, roberta_valid, vgg16_valid, y_valid, \
user_id_test, item_id_test, roberta_test, vgg16_test, y_test = data_split(df)

# Model

In [ ]:
def CAMRec(num_users, num_items, dropout, learning_rate):

    # 사용자 ID
    user_id_input = Input(shape=(1,), name='user_id_input')
    user_emb = Embedding(input_dim=num_users, output_dim=128, name='user_embedding')(user_id_input)
    user_emb = Flatten()(user_emb)

    # 아이템 ID
    item_id_input = Input(shape=(1,), name='item_id_input')
    item_emb = Embedding(input_dim=num_items, output_dim=128, name='item_embedding')(item_id_input)
    item_emb = Flatten()(item_emb)

    # Text Input
    roberta_input = Input(shape=(768,), name='roberta_input')

    # Image Input
    vgg16_input = Input(shape=(4096,), name='vgg16_input')

    # Interaction MLP
    ncf_nlp = Concatenate()([user_emb, item_emb])
    ncf_nlp = Dense(128, activation='relu')(ncf_nlp)
    ncf_nlp = Dropout(dropout)(ncf_nlp)
    ncf_nlp = Dense(64, activation='relu')(ncf_nlp)
    ncf_nlp = Dropout(dropout)(ncf_nlp)

    # Text MLP
    roberta_nlp = Dense(512, activation='relu')(roberta_input)
    roberta_nlp = Dropout(dropout)(roberta_nlp)
    roberta_nlp = Dense(256, activation='relu')(roberta_nlp)
    roberta_nlp = Dropout(dropout)(roberta_nlp)

    # Image MLP
    vgg16_nlp = Dense(512, activation='relu')(vgg16_input)
    vgg16_nlp = Dropout(dropout)(vgg16_nlp)
    vgg16_nlp = Dense(256, activation='relu')(vgg16_nlp)
    vgg16_nlp = Dropout(dropout)(vgg16_nlp)

    # Co-Attention Input
    roberta_vec = Lambda(lambda x: tf.expand_dims(x, axis=1), name='roberta_expand')(roberta_nlp)
    vgg16_vec = Lambda(lambda x: tf.expand_dims(x, axis=1), name='vgg16_expand')(vgg16_nlp)

    # Co-Attention(bi-directional cross attention)
    text_attention = MultiHeadAttention(num_heads=4, key_dim=64, name='roberta_cross_attention')(query=roberta_vec, value=vgg16_vec, key=vgg16_vec)
    img_attention = MultiHeadAttention(num_heads=4, key_dim=64, name='vgg16_cross_attention')(query=vgg16_vec, value=roberta_vec, key=roberta_vec)

    text_attention = Lambda(lambda x: tf.squeeze(x, axis=1), name='roberta_squeeze')(text_attention) # [batch, emb]
    img_attention = Lambda(lambda x: tf.squeeze(x, axis=1), name='vgg16_squeeze')(img_attention) # [batch, emb]

    multi_modal_rep = Concatenate(axis=-1)([text_attention, img_attention]) # [batch, emb*2]

    # Co-Attention MLP
    multi_modal_rep = Dense(256, activation='relu')(multi_modal_rep)
    multi_modal_rep = Dropout(dropout)(multi_modal_rep)

    # Final
    prediction = Concatenate(name='all_representation')([ncf_nlp, multi_modal_rep])
    prediction = Dense(64, activation='relu')(prediction)
    final = Dense(1, activation='linear', name='rating_prediction')(prediction)

    model = Model(inputs=[user_id_input, item_id_input, roberta_input, vgg16_input],
                  outputs=final)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mean_squared_error', metrics=['mean_absolute_error', 'mean_squared_error'])

    return model

In [ ]:
dropout = 0.0
learning_rate = 0.001

In [ ]:
model = CAMRec(n_users, n_items, dropout, learning_rate)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ vgg16_input         │ (None, 4096)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ roberta_input       │ (None, 768)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 512)       │  2,097,664 │ vgg16_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 512)       │    393,728 │ roberta_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 512)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 512)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_id_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_id_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 256)       │    131,328 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256)       │    131,328 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 128)    │    953,472 │ user_id_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 128)    │  2,060,672 │ item_id_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 256)       │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 128)       │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 128)       │          0 │ item_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16_expand        │ (None, 1, 256)    │          0 │ dropout_5[0][0]   │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ roberta_expand      │ (None, 1, 256)    │          0 │ dropout_3[0][0]   │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ roberta_cross_atte… │ (None, 1, 256)    │    263,168 │ vgg16_expand[0][… │
│ (MultiHeadAttentio… │                   │            │ roberta_expand[0

 Total params: 6,487,617 (24.75 MB)

 Trainable params: 6,487,617 (24.75 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = EarlyStopping(monitor ='val_loss', min_delta = 0.001, patience = 5, verbose = 1, mode = 'min', restore_best_weights = True)

In [ ]:
history = model.fit([user_id_train, item_id_train, roberta_train, vgg16_train],
                    y_train,
                    validation_data = ([user_id_valid, item_id_valid, roberta_valid, vgg16_valid], y_valid),
                    epochs = 100,
                    batch_size = 256,
                    callbacks=[early_stopping])

Epoch 1/100
183/183 ━━━━━━━━━━━━━━━━━━━━ 41s 123ms/step - loss: 8.4631 - mean_absolute_error: 1.4639 - mean_squared_error: 8.4631 - val_loss: 0.3353 - val_mean_absolute_error: 0.4115 - val_mean_squared_error: 0.3353
Epoch 2/100
183/183 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.2853 - mean_absolute_error: 0.3893 - mean_squared_error: 0.2853 - val_loss: 0.3012 - val_mean_absolute_error: 0.4085 - val_mean_squared_error: 0.3012
Epoch 3/100
183/183 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1701 - mean_absolute_error: 0.2952 - mean_squared_error: 0.1701 - val_loss: 0.3179 - val_mean_absolute_error: 0.4266 - val_mean_squared_error: 0.3179
Epoch 4/100
183/183 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0826 - mean_absolute_error: 0.2064 - mean_squared_error: 0.0826 - val_loss: 0.3395 - val_mean_absolute_error: 0.4196 - val_mean_squared_error: 0.3395
Epoch 5/100
183/183 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0516 - mean_absolute_error: 0.1684 - mean_squared_error: 0.0516 - val_loss: 0.3118

In [ ]:
# 모델 성능 측정
predicted_ratings = model.predict([user_id_test, item_id_test, roberta_test, vgg16_test])

MAE = mean_absolute_error(y_test, predicted_ratings)
MSE = mean_squared_error(y_test, predicted_ratings)
RMSE = np.sqrt(MSE)
MAPE = mean_absolute_percentage_error(y_test, predicted_ratings) * 100


print(f'MAE: {MAE:.4f}')
print(f'MSE: {MSE:.4f}')
print(f'RMSE: {RMSE:.4f}')
print(f'MAPE: {MAPE:.4f}%')

417/417 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
MAE: 0.4076
MSE: 0.3045
RMSE: 0.5518
MAPE: 11.2622%
